# 04 - Ollama LLM classifier via LangChain

Use a local Ollama model (`llama3.1:8b`) through LangChain to
classify the same 67 test-set stories that notebook 03 evaluated
the TF-IDF baseline on. Then compare the two head-to-head.

**What you will learn**

- How to instantiate a LangChain `ChatOllama` model against a
  locally-running Ollama daemon and route classification through
  it.
- How to structure a classification prompt that produces clean
  single-token category outputs.
- How LLM-in-the-loop classification compares to a linear
  baseline when the label set is small (7 categories) and the
  descriptions are well-defined.
- The latency and token-cost profile of local 8B inference vs a
  cloud API - this project runs entirely offline on the GPU.

## Setup

In [1]:
%load_ext autoreload
%autoreload 2

import sys
import time
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

from src.llm import get_chat_ollama
from src.classify import LLMClassifier
from src.evaluate import accuracy, per_class_report
from src.labels import category_names

pd.set_option("display.width", 120)
pd.set_option("display.max_colwidth", 90)
np.random.seed(0)

## 1. Reproduce the exact nb03 test split

Same seed, same stratification, same silver labels. If we drop this
we cannot make a fair head-to-head comparison in section 5.

In [2]:
stories = pd.read_csv(ROOT / "data" / "stories_ai.csv")
silver = pd.read_csv(ROOT / "data" / "silver_labels.csv")
df = stories.merge(silver, on="id", how="inner")
df["title"] = df["title"].fillna("")
df["text"] = df["text"].fillna("")

train, test = train_test_split(
    df, test_size=0.30, random_state=0, stratify=df["category"],
)
print(f"test rows: {len(test)}")
print("class distribution in test:")
print(test["category"].value_counts().to_string())

test rows: 67
class distribution in test:
category
opinion     19
product     18
news        11
research     7
tutorial     6
tool         5
other        1


## 2. Instantiate the LangChain classifier

`src.classify.LLMClassifier` wraps a LangChain chat model with the
classification prompt from `CLASSIFY_PROMPT_TEMPLATE` in
`src/classify.py`. It:

- Injects the category descriptions from `src/labels.py` into the
  prompt (so both nb02's qwen labeller and this llama classifier
  read from the same source of truth).
- Truncates the story body to 1500 chars to avoid over-long
  prompts.
- Parses the response's first line, snaps it to the closest known
  category, and defaults to `other` if the LLM went off-schema.

In [3]:
CLASSIFIER_MODEL = "llama3.1:8b"
chat = get_chat_ollama(model=CLASSIFIER_MODEL)
clf = LLMClassifier(chat=chat)
print(f"classifier: {CLASSIFIER_MODEL}")

classifier: llama3.1:8b


## 3. Predict on the test set

This calls the local Ollama daemon 67 times. On a laptop GPU each
call is ~1-2 seconds so total ~1-3 minutes. Progress printed every
10 items.

In [4]:
t0 = time.time()
titles = test["title"].tolist()
bodies = test["text"].tolist()

preds = []
for i, (t, b) in enumerate(zip(titles, bodies)):
    preds.append(clf._predict_one(t, b))
    if (i + 1) % 10 == 0:
        rate = (i + 1) / (time.time() - t0)
        print(f"  {i + 1}/{len(titles)}  ({rate:.2f} items/s)")

elapsed = time.time() - t0
print(f"done: {len(preds)} predictions in {elapsed:.1f}s "
      f"({len(preds) / elapsed:.2f} items/s)")

  10/67  (0.68 items/s)


  20/67  (1.00 items/s)


  30/67  (1.19 items/s)


  40/67  (1.32 items/s)


  50/67  (1.43 items/s)


  60/67  (1.50 items/s)


done: 67 predictions in 43.6s (1.54 items/s)


## 4. Evaluate against silver labels

In [5]:
y_test = test["category"].tolist()
acc = accuracy(y_test, preds)
print(f"LLM classifier accuracy vs silver: {acc * 100:.1f}%  ({len(y_test)} rows)")
print()
report = per_class_report(y_test, preds, categories=category_names())
print(report.round(3).to_string())

LLM classifier accuracy vs silver: 62.7%  (67 rows)

          support  precision  recall     f1
category                                   
research        7      0.438   1.000  0.609
product        18      1.000   0.556  0.714
tool            5      0.385   1.000  0.556
tutorial        6      0.375   0.500  0.429
opinion        19      1.000   0.263  0.417
news           11      0.846   1.000  0.917
other           1      0.500   1.000  0.667


## 5. Head-to-head with the TF-IDF baseline

Load the nb03 predictions on the same test set and compare
row-by-row: where does each method agree with silver, and where do
they disagree with each other?

In [6]:
tfidf = pd.read_csv(ROOT / "data" / "predictions_tfidf.csv")
tfidf = tfidf.set_index("id")

merged = pd.DataFrame({
    "id":        test["id"].values,
    "title":     titles,
    "silver":    y_test,
    "tfidf":     [tfidf.loc[i, "tfidf_pred"] for i in test["id"].values],
    "llm":       preds,
})
merged["tfidf_correct"] = merged["tfidf"] == merged["silver"]
merged["llm_correct"]   = merged["llm"]   == merged["silver"]

print(f"TF-IDF accuracy: {merged['tfidf_correct'].mean() * 100:.1f}%")
print(f"LLM accuracy:    {merged['llm_correct'].mean() * 100:.1f}%")
print()
print("agreement breakdown:")
print(f"  both correct:      {int((merged['tfidf_correct'] & merged['llm_correct']).sum()):>3}")
print(f"  only TF-IDF right: {int((merged['tfidf_correct'] & ~merged['llm_correct']).sum()):>3}")
print(f"  only LLM right:    {int((~merged['tfidf_correct'] & merged['llm_correct']).sum()):>3}")
print(f"  both wrong:        {int((~merged['tfidf_correct'] & ~merged['llm_correct']).sum()):>3}")

TF-IDF accuracy: 43.3%
LLM accuracy:    62.7%

agreement breakdown:
  both correct:       13
  only TF-IDF right:  16
  only LLM right:     29
  both wrong:          9


## 6. Where does the LLM win?

The interesting rows are where **LLM is right and TF-IDF is wrong** -
that is the lift the LLM adds. Look at a sample.

In [7]:
llm_wins = merged[merged['llm_correct'] & ~merged['tfidf_correct']]
tfidf_wins = merged[merged['tfidf_correct'] & ~merged['llm_correct']]
print(f"LLM-only wins: {len(llm_wins)} rows")
print(llm_wins[["id", "title", "silver", "tfidf", "llm"]].head(10).to_string(index=False))
print()
print(f"TF-IDF-only wins: {len(tfidf_wins)} rows")
print(tfidf_wins[["id", "title", "silver", "tfidf", "llm"]].head(10).to_string(index=False))

LLM-only wins: 29 rows
      id                                                                      title   silver    tfidf      llm
49319389  Research papers using "kidney disappointment" instead of "kidney failure" research  opinion research
49333151               LLM City – 3D render of all Kimi K3's weights as 2.5mm tiles     tool research     tool
49363035                           AI Used to Verify Toughest Mathematics Proof Yet research  opinion research
49285160                                                DeepSeek API Pricing Update     news     tool     news
49335009        Apple AirTag reveals how Amazon destroys rare books for AI training     news  opinion     news
49362322          Prompt caching makes self-consistency cheap for long-context LLMs research  product research
49274600                                                       DeepSeek V4 Pro 0813  product     tool  product
49362909                             A pocketable AI agent – Tokens free to use now  prod

**Reading the wins.** The LLM should tend to win on stories
where the *semantics* matter (a title like "How X works" needs
understanding, not just keyword matching), and the TF-IDF baseline
should win where a single strong lexical cue exists ("Show HN:"
prefix, "arxiv.org" URL). If instead the LLM wins on a random-
looking mix and TF-IDF has almost no wins, the LLM is
substantially better on this problem; if the reverse, the linear
model already captures the signal.

## 7. Cross-classifier confusion

Where do the two methods disagree with each other, regardless of
what silver says? A cross-tab tells us which category pairs the
LLM and TF-IDF systematically see differently.

In [8]:
cross = pd.crosstab(merged['tfidf'], merged['llm'], margins=True, margins_name="all")
print("rows: TF-IDF prediction, cols: LLM prediction")
print(cross.to_string())

rows: TF-IDF prediction, cols: LLM prediction
llm       news  opinion  other  product  research  tool  tutorial  all
tfidf                                                                 
news         4        1      0        0         1     2         1    9
opinion      6        0      0        1         9     0         4   20
product      1        1      0        8         2     6         0   18
research     1        1      1        0         1     2         0    6
tool         1        1      1        1         1     2         1    8
tutorial     0        1      0        0         2     1         2    6
all         13        5      2       10        16    13         8   67


## 8. Save LLM predictions for downstream comparison

In [9]:
out_path = ROOT / "data" / "predictions_llm.csv"
merged.to_csv(out_path, index=False)
print(f"saved {len(merged)} test-set rows to {out_path}")

saved 67 test-set rows to C:\Users\anjan\Desktop\Goals\github\hn-ml-trends\data\predictions_llm.csv


## Takeaways for notebook 05

- We now have three per-row signals on the 67-story test set:
  the silver label, the TF-IDF prediction, and the LLM prediction.
- The absolute accuracy numbers should both be interpreted against
  the ~62.5% silver-label ceiling from nb02 - a method scoring
  around or above that number is essentially at the silver-label
  quality wall, not miles behind a perfect oracle.
- The interesting signal is *shape*: which category pairs each
  method confuses. Both should confuse the fuzzy pairs
  (product / tool, tutorial / research); if only one does, that
  points to a real capability difference.
- Notebook 05 turns from classification to **clustering**:
  embed all 223 stories with a sentence-transformer, cluster them
  via KMeans and HDBSCAN, and check how well the clusters line up
  with the silver labels. Clusters can *find* themes the fixed
  category set misses; the two views (labels + clusters) will feed
  the LangGraph agent in nb06.